# 多卡并行：单模型 vs 单卡（`parallel_compute`）

## 能否「一个 model + 四张卡」？

**可以。** 本 notebook 使用 **`torch.nn.DataParallel`**（单进程、**一份权重**）：

- 每个 training step 取 **`bg_batch=4`** 个 z-slice；
- `bg_sample_mode=z_shard`：四个 slice 分别来自 z 的四个 **128 层分片**（同一步并行前向）；
- 梯度在 GPU0 上聚合后更新 **同一个** Micro-UNet。
- 推理/评测仍是 **单模型** `run_bg_inference`（**1×** NN 参数量 → effective CR 与单卡一致）。

| 方案 | 模型数 | NN 存储 | 训练 |
|------|--------|---------|------|
| **本 notebook（DP）** | **1** | 1× | DataParallel + z_shard |
| 四卡各训 15s 再拼接 | 4 | 4× | 四个独立 checkpoint（见旧讨论，**未**在此实现） |
| DDP（多进程） | 1 | 1× | 需 `torchrun`，notebook 外脚本更合适 |

## 对比实验

| 模式 | 说明 |
|------|------|
| `single_60` | 1 GPU，`bg_batch=1`，`sequential`，60s |
| `dp4_60` | 4 GPU DataParallel，`bg_batch=4`，`z_shard`，60s |
| `dp4_15` | 同上，15s（墙钟约 1/4，可与「四卡各 15s 四模型」墙钟对照） |

> 需要 `bg_stage.py` 中 `cfg.bg_data_parallel=True` 与 `z_shard` 采样（已接入）。


In [ ]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.append("/home/sam/Halo_Finder/Final_design/base_script")

from config_io import load_multifield_from_disk
from experiment import build_bg_only_cfg, estimate_bg_model_param_bytes
from bg_stage import run_bg_inference, train_bg_only, unwrap_bg_model


def _global_diag(x_true, x_hat):
    """Global reconstruction metrics (replaces the old ROI diagnostics)."""
    x_true = np.asarray(x_true); x_hat = np.asarray(x_hat)
    dr = float(x_true.max() - x_true.min()) or 1.0
    mse = float(np.mean((x_true - x_hat) ** 2))
    psnr = 20 * np.log10(dr) - 10 * np.log10(mse + 1e-12) if mse > 0 else 100.0
    max_err = float(np.max(np.abs(x_true - x_hat)))
    return {"psnr": psnr, "max_err": max_err}

pysz_path = r"/home/sam/Data_Compression/SZ3/tools/pysz"
if pysz_path not in sys.path:
    sys.path.append(pysz_path)
from pysz import SZ


def set_seed(seed=17):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


NUM_GPUS = int(torch.cuda.device_count()) if torch.cuda.is_available() else 0
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"cuda devices: {NUM_GPUS} | train device: {device}")


In [ ]:
base_path = Path(r"/home/sam/Halo_Finder/halo_finder_v1/SDRBENCH-EXASKY-NYX-512x512x512/origin").resolve().as_posix() + "/"
sz_lib_path = r"/home/sam/Data_Compression/SZ3/build/lib64/libSZ3c.so"
data_shape = (512, 512, 512)
TARGET_STEM = "dark_matter_density"
FIELD_FILES = [
    "dark_matter_density.f32", "velocity_z.f32", "baryon_density.f32",
    "temperature.f32", "velocity_x.f32", "velocity_y.f32",
]
REL_SETTINGS = [
    ("r0", 1e-4), ("r1", 2e-4), ("r2", 3e-4), ("r3", 4e-4),
    ("r4", 5e-4), ("r5", 6e-4), ("r6", 1e-5),
]
REL_PROBE = REL_SETTINGS[0][1]

fname = TARGET_STEM + ".f32"
gt_path = base_path + fname
aux_paths = [base_path + f for f in FIELD_FILES if f != fname]


def rel_sz_suffix(rel_err):
    return f"{rel_err:.0e}".replace("+", "")


sz_bin_path = base_path + Path(fname).stem + "_rel" + rel_sz_suffix(REL_PROBE) + ".sz"
if not Path(sz_bin_path).is_file():
    eng = SZ(sz_lib_path)
    vol = np.fromfile(gt_path, dtype=np.float32).reshape(data_shape)
    Path(sz_bin_path).write_bytes(eng.compress(vol, 1, 0, REL_PROBE, 0)[0])

Xs, _ = load_multifield_from_disk(
    gt_path=gt_path, aux_paths=aux_paths, sz_bin_path=sz_bin_path,
    data_shape=data_shape, pysz_path=pysz_path, sz_lib_path=sz_lib_path,
)

sz = SZ(sz_lib_path)
gt_target = np.asarray(Xs[0], np.float32)
aux_fields_gt = [np.asarray(f, np.float32) for f in Xs[1:]]
ORIGINAL_TARGET_BYTES = int(gt_target.nbytes)
N_IN = 6
D, H, W = Xs[0].shape


def build_Xps_for_rel(rel_err, aux_sz=False):
    rel_err = float(rel_err)
    b_dm, cr_dm = sz.compress(gt_target, 1, 0, rel_err, 0)
    x_lq_dm = sz.decompress(b_dm, gt_target.shape, np.float32)
    return [x_lq_dm] + aux_fields_gt, float(cr_dm), b_dm, 0


set_seed(17)


In [ ]:
# ---- 与 laplacian global 对齐的固定超参 ----
BG_LR = 1e-3
BG_ARCH = "spatial"
BG_H = 7
BG_PATCH = 512
MODEL_DTYPE_BYTES = 2

# 训练模式：改 MODE 或一次跑 COMPARE_MODES 里全部
MODE_SPECS = {
    "single_60": {
        "label": "1 GPU | batch=1 | sequential | 60s",
        "max_train_time": 60.0,
        "bg_batch": 1,
        "bg_sample_mode": "sequential",
        "bg_data_parallel": False,
        "steps_per_epoch": int(D),
    },
    "dp4_60": {
        "label": f"4 GPU DP | batch={max(NUM_GPUS, 4)} | z_shard | 60s",
        "max_train_time": 60.0,
        "bg_batch": max(NUM_GPUS, 4),
        "bg_sample_mode": "z_shard",
        "bg_data_parallel": True,
        "steps_per_epoch": max(D // max(NUM_GPUS, 4), 1),
    },
    "dp4_15": {
        "label": f"4 GPU DP | batch={max(NUM_GPUS, 4)} | z_shard | 15s",
        "max_train_time": 15.0,
        "bg_batch": max(NUM_GPUS, 4),
        "bg_sample_mode": "z_shard",
        "bg_data_parallel": True,
        "steps_per_epoch": max(D // max(NUM_GPUS, 4), 1),
    },
}

COMPARE_MODES = ["single_60", "dp4_60"]  # 可加 "dp4_15"
REL_PROBE = REL_SETTINGS[0][1]

n_p, _ = estimate_bg_model_param_bytes(
    n_fields=int(N_IN),
    shape=data_shape,
    bg_arch=BG_ARCH,
    bg_h=int(BG_H),
    dtype_bytes=MODEL_DTYPE_BYTES,
    bg_split_bands=True,
    bg_split_mode="three",
)
print(f"Micro-UNet params (bg_h={BG_H}): {n_p:,}  (~{n_p * MODEL_DTYPE_BYTES} bytes BF16)")

if NUM_GPUS < 2:
    print("警告: 检测到 GPU<2，dp4_* 模式会退化为单卡（仍为一个 model）。")


In [ ]:
def psnr_np(x_true, x_hat):
    x_true, x_hat = np.asarray(x_true, np.float64), np.asarray(x_hat, np.float64)
    mse = float(np.mean((x_true - x_hat) ** 2))
    dr = float(np.max(x_true) - np.min(x_true))
    if mse <= 0:
        return float("inf")
    return 20.0 * np.log10(dr) - 10.0 * np.log10(mse)


def model_param_bytes(model):
    return int(sum(p.numel() for p in model.parameters()) * MODEL_DTYPE_BYTES)


def effective_cr_dm(sz3_bytes, nn_bytes):
    return float(ORIGINAL_TARGET_BYTES / max(float(sz3_bytes) + float(nn_bytes), 1.0))


def build_cfg_for_mode(mode_spec, rel_err, Xps_list):
    rel_err = float(rel_err)
    cfg = build_bg_only_cfg(
        X_target=Xs[0],
        Xps=Xps_list,
        max_train_time=float(mode_spec["max_train_time"]),
        bg_h=int(BG_H),
        roi_h=4,
        epochs=200,
        steps_per_epoch=int(mode_spec["steps_per_epoch"]),
        bg_patch_size=int(BG_PATCH),
        bg_batch=int(mode_spec["bg_batch"]),
        lr=float(BG_LR),
    )
    cfg.bg_arch = BG_ARCH
    cfg.bg_split_mode = "three"
    cfg.bg_split_bands = True
    cfg.bg_sample_mode = str(mode_spec["bg_sample_mode"])
    cfg.bg_data_parallel = bool(mode_spec["bg_data_parallel"])
    cfg.bg_log_prefix = str(mode_spec.get("mode_id", "run"))
    cfg.bg_input_freq_aug = False
    cfg.use_hint = False
    cfg.bg_input_norm = "global"
    cfg.bg_residual_norm = "global"
    cfg.bg_freq_weight = 1.0
    cfg.bg_freq_focus = "low"
    cfg.rel_err = rel_err
    return cfg


def train_mode(mode_id, rel_err=REL_PROBE):
    mode_spec = {**MODE_SPECS[mode_id], "mode_id": mode_id}
    rel_err = float(rel_err)
    Xps_base, _, b, _ = build_Xps_for_rel(rel_err, aux_sz=False)
    Xps_list = [np.asarray(f, np.float32) for f in Xps_base[:N_IN]]
    cfg = build_cfg_for_mode(mode_spec, rel_err, Xps_list)

    print(
        f"\n{'='*64}\n  {mode_spec['label']}\n"
        f"  batch={cfg.bg_batch} sample={cfg.bg_sample_mode} "
        f"data_parallel={cfg.bg_data_parallel} steps/ep={cfg.steps_per_epoch}\n{'='*64}"
    )

    ep_patch, ep_k = 32, 7
    first = [True]

    def evaluator(m):
        m = unwrap_bg_model(m)
        if first[0]:
            first[0] = False
            m2 = _global_diag(Xs[0], Xps_list[0])
            return m2["psnr"], m2["max_err"]
        xh = run_bg_inference(m, Xs, Xps_list, cfg, rel_err)
        m2 = _global_diag(Xs[0], xh)
        return m2["psnr"], m2["max_err"]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    t0 = time.perf_counter()
    model, hist = train_bg_only(Xs=Xs, Xps=Xps_list, device=device, cfg=cfg, evaluator=evaluator)
    wall = time.perf_counter() - t0
    model = unwrap_bg_model(model)

    x_hat = np.asarray(run_bg_inference(model, Xs, Xps_list, cfg, rel_err), np.float32)
    return {
        "mode": mode_id,
        "label": mode_spec["label"],
        "train_epochs": int(len(hist.get("loss") or [])),
        "wall_s": float(wall),
        "rel_err": rel_err,
        "psnr_512": float(psnr_np(gt_target, x_hat)),
        "nn_bytes": model_param_bytes(model),
        "effective_cr": effective_cr_dm(len(b), model_param_bytes(model)),
        "bg_batch": int(cfg.bg_batch),
        "data_parallel": bool(cfg.bg_data_parallel),
    }


## Probe 对比（`REL_PROBE`）


In [ ]:
rows = []
for mode_id in COMPARE_MODES:
    if mode_id.startswith("dp4") and NUM_GPUS < 2:
        print(f"跳过 {mode_id}：需要 >=2 张 GPU")
        continue
    set_seed(17)
    rows.append(train_mode(mode_id, rel_err=REL_PROBE))

df_parallel = pd.DataFrame(rows)
display(df_parallel[["mode", "label", "train_epochs", "wall_s", "psnr_512", "effective_cr", "nn_bytes"]])

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].bar(df_parallel["mode"], df_parallel["psnr_512"], color=["#4c72b0", "#55a868", "#c44e52"][: len(df_parallel)])
axes[0].set_ylabel("PSNR @ 512³ (dB)")
axes[0].set_title(f"rel={REL_PROBE}")
axes[0].grid(True, axis="y", alpha=0.35)

axes[1].bar(df_parallel["mode"], df_parallel["wall_s"], color=["#4c72b0", "#55a868", "#c44e52"][: len(df_parallel)])
axes[1].set_ylabel("wall time (s)")
axes[1].set_title("train+eval wall")
plt.tight_layout()
plt.show()


## 说明：如何解读结果

- **DP 模式仍是 1 个 model**：`nn_bytes` 应与 `single_60` 相同（约 3763×2 B）。
- **每 step 吞吐**：`z_shard` + `batch=4` 每步覆盖四个 z 分片各一层；`steps_per_epoch=128` 时一个 epoch 扫完 512 个 z（与 sequential 512 步等价，但每步 4 路并行）。
- **60s 对比**：若 DP 墙钟更短、PSNR 接近，说明多卡加速了 **同权重** 训练；若 PSNR 明显不同，可能是 batch/采样分布变化，而非「四个模型拼接」。
- **真·四模型拼接**（4× NN CR）需另开实验；本 notebook 刻意避免。
